In [ ]:
# !pip install langchain-text-splitters

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from transformers import pipeline
from tqdm import tqdm
import re

In [3]:
model = SentenceTransformer('all-MiniLM-L6-v2')

In [4]:
def chunking_function(fp, chunks_function):
    with open(fp, 'r', encoding='utf-8') as file:
        text = file.read()
        chunks = chunks_function(text)
        embeddings = model.encode(chunks)
    return chunks, embeddings

In [5]:
file_path = "D:\\LLM-Learning-Journey\\LLM-Learning-Journey\\Week3 - Embedding and vector Database\\Data\\Business_Review.txt"

## Different chunking

### 1. Fixed length chunking

In [6]:
def fixed_length_chunking(text, chunk_size=512):
    return [text[i:i+chunk_size] for i in range(0, len(text), chunk_size)]

In [7]:
# with open("D:\\LLM-Learning-Journey\\LLM-Learning-Journey\\Week3 - Embedding and vector Database\\Data\\HR_Employee_benefit.txt", 'r') as file:
#     text = file.read()
# Fc_chunks = fixed_length_chunking(text, chunk_size=512)
# Fc_chunk_embeddings = model.encode(Fc_chunks)
Fc_chunks, Fc_chunk_embeddings = chunking_function(file_path, fixed_length_chunking)

In [8]:
len(Fc_chunks)

5

### 2. Context Aware Chunking

In [9]:
def content_aware_chunking(text):
    paragraphs = text.split("\n\n")  # split by paragraph
    chunks = [p.strip() for p in paragraphs if p.strip()]
    return chunks

In [10]:
# with open("D:\\LLM-Learning-Journey\\LLM-Learning-Journey\\Week3 - Embedding and vector Database\\Data\\HR_Employee_benefit.txt", 'r') as file:
#     text = file.read()
# cac_chunks = content_aware_chunking(text)
# cac_chunk_embeddings = model.encode(cac_chunks)
cac_chunks, cac_chunk_embeddings = chunking_function(file_path, content_aware_chunking)

In [11]:
len(cac_chunks)

10

### 3. Recursive Character Chunking

In [12]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
def recursive_chunking(text, chunk_size=512, chunk_overlap=50):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap, separators=["\n\n", "\n", " ", ""])
    chunks = text_splitter.split_text(text)
    return chunks

In [13]:
rc_chunks, rc_chunk_embeddings = chunking_function(file_path, recursive_chunking)

In [14]:
len(rc_chunks)

7

### 4. Document Structure-Based Chunking

In [15]:
import re

def structure_based_chunking(text):
    pattern = r"\nChapter |\nSection |\n\d+\.\s|\n"
    sections = re.split( pattern,text)  # split by headings
    chunks = []
    
    for sec in sections:
        sec = sec.strip()
        if sec:
            chunks.append(sec)
    
    return chunks

In [16]:
DSC_chunks, DSC_chunk_embeddings = chunking_function(file_path, structure_based_chunking)

In [17]:
len(DSC_chunks)

39

### 5. Semantic Chunking

In [18]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import re

def semantic_chunk(text, model_name='all-MiniLM-L6-v2', percentile_threshold=85):
    model = SentenceTransformer(model_name)
    sentences = [s.strip() for s in re.split(r'(?<=[.!?])\s+', text) if s.strip()]
    if len(sentences) < 2: # Can't find a break if there's only 1 sentence
        return [text]
    embeddings = model.encode(sentences)
    similarities = []
    for i in range(len(embeddings) - 1):
        sim = cosine_similarity([embeddings[i]], [embeddings[i+1]])[0][0]
        similarities.append(sim)
    threshold = np.percentile(similarities, 100 - percentile_threshold)
    breakpoints = [i for i, sim in enumerate(similarities) if sim < threshold]
    chunks = []
    start_idx = 0
    for bp in breakpoints:
        # bp is the index of the *last* sentence in the chunk
        chunk = ' '.join(sentences[start_idx : bp + 1])
        chunks.append(chunk)
        start_idx = bp + 1 # next chunk starts after the break
    chunks.append(' '.join(sentences[start_idx:]))

    return chunks

In [19]:
sc_chunks, sc_chunk_embeddings = chunking_function(
    file_path, semantic_chunk)

In [20]:
len(sc_chunks)

8

### Contextual Chunking

In [21]:
LLM = pipeline("text-generation",
               model="microsoft/Phi-3-mini-4k-instruct",
               device="cpu", # "cuda" if GPU
               dtype="auto")

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Device set to use cpu


In [22]:
def need_contextual_chunking(chunk):
    return bool(re.match(r"It|this|that|these|those|he|she|they|his|her|their|its|my|your|our", chunk.strip(), re.IGNORECASE))

In [35]:
def summarize_chunk(current_chunk, previous_Chunk, model=LLM):
    messages = [
        {"role": "system", "content": "You are a helpful to identify the pronouns in the current chunk and determine if it needs the previous chunk as context to be understood."},
        {"role": "user", "content": f"Current chunk: '{current_chunk}' Previous chunk: '{previous_Chunk}' Does the current chunk contain pronouns that refer to entities in the previous chunk? If so, summarize the previous chunk to provide context for understanding the current chunk., if not respond with None."}
    ]
    LLM_response = model(messages, max_length=200, num_return_sequences=1)
    # return LLM_response
    # print(LLM_response[0]['generated_text'])
    summary = LLM_response[0]['generated_text'][2]["content"].strip()
    return summary


In [36]:
def contextual_chunking(text):
    chunks = semantic_chunk(text)
    contextual_chunks = []
    for i, chunks in tqdm(enumerate(chunks), total=len(chunks)):
        print(f"{i}. {need_contextual_chunking(chunks)}")
        if i == 0:
            contextual_chunks.append(chunks)
        else:
            if need_contextual_chunking(chunks):
                print(f"{i}. Chunk: '{chunks}' needs context from previous chunk: '{contextual_chunks[-1]}'")
                summary = summarize_chunk(chunks, contextual_chunks[-1])
                if summary and summary.lower() != "none":
                    contextual_chunks.append("Context: " + summary + "  actual chunk: " + chunks)
                else:
                    contextual_chunks.append(chunks)
            else:
                contextual_chunks.append(chunks)
    return contextual_chunks

In [37]:
contextual_chunks, contextual_chunk_embeddings = chunking_function(file_path, contextual_chunking)

  0%|          | 0/8 [00:00<?, ?it/s]Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


0. False
1. False
2. False
3. False
4. False
5. False
6. True
6. Chunk: 'They are primarily focused on the data platform. It is critical for Q4 goals. 4. Risks and Mitigation
- Risk 1: Cloud costs. Mitigation: Finish migration, add query caps. - Risk 2: Hiring competition. Mitigation: Increase referral bonus to $10k. - Risk 3: SMB churn. It improved in Q3 but remains a watch item. 5. Appendix: Key Metrics Table
Metric         | Q2 2025 | Q3 2025 | Change
Revenue        | $1.78M  | $2.10M  | +18%
DAU            | 1,500   | 2,000   | +33%
Cloud Spend    | $323k   | $420k   | +30%
NPS            | 42      | 51      | +9

Next steps: The leadership team will meet offsite in Denver next month. They will finalize the 2026 roadmap there.' needs context from previous chunk: '3.3 Headcount
We hired 12 engineers in Q3. The total team size is now 84.'


100%|██████████| 8/8 [01:08<00:00,  8.61s/it]


7. False


Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


" Yes, the current chunk contains pronouns that refer to entities in the previous chunk. The pronoun 'They' in the sentence 'The leadership team will meet offsite in Denver next month. They will finalize the 2026 roadmap there' refers to the 'leadership team' mentioned in the previous chunk.\n\nSummary of the previous chunk for context:\nIn the previous chunk, it is stated that '3.3 Headcount: We hired 12 engineers in Q3. The total team size is now 84.' This indicates that the company has recently expanded its engineering team by hiring 12 new engineers, bringing the total team size to 84 members. This context is crucial for understanding that the leadership team referred to in the next chunk will be responsible for finalizing the company's roadmap for the next year, which will likely involve considerations about the newly expanded engineering team's contributions and strategies."